In [12]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
import random
from time import sleep
import pandas as pd
from dotenv import load_dotenv
import os
import requests
import unicodedata

In [13]:
load_dotenv()
WEB_BASE = os.getenv("WEB_BASE")
WEB_MANUFACTURER = os.getenv("WEB_MANUFACTURER")
WEB_MODELS = os.getenv("WEB_MODELS")
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json"
}

In [14]:
url_marcas = WEB_MANUFACTURER
print(url_marcas)

https://www.compramostucoche.es/papi/v1/car-types/manufacturer


In [26]:
marcas_json = requests.get(url_marcas, headers=headers).json()
marcas = marcas_json.get('wkda', {})

catalogo = []

for id, nombre_marca in marcas.items():
    url_modelos = f"{WEB_MODELS}{id}"
    modelos_marca = requests.get(url_modelos, headers=headers).json()
    modelos = modelos_marca.get('wkda', {})
    
    marca = {
        "id_marca": id,
        "nombre": nombre_marca,
        "modelos": modelos,
        "total_modelos": len(modelos)
    }

    catalogo.append(marca)

    sleep(1)
    print(f"añadidos {marca['total_modelos']} modelos de la marca {marca['nombre']}")

añadidos 13 modelos de la marca Abarth
añadidos 2 modelos de la marca Aiways
añadidos 26 modelos de la marca Alfa Romeo
añadidos 11 modelos de la marca Alpina
añadidos 2 modelos de la marca Alpine
añadidos 9 modelos de la marca Aston Martin
añadidos 79 modelos de la marca Audi
añadidos 2 modelos de la marca Barkas
añadidos 4 modelos de la marca Bentley
añadidos 29 modelos de la marca BMW
añadidos 10 modelos de la marca BYD
añadidos 15 modelos de la marca Cadillac
añadidos 1 modelos de la marca Caterham
añadidos 24 modelos de la marca Chevrolet
añadidos 18 modelos de la marca Chrysler
añadidos 55 modelos de la marca Citroen
añadidos 3 modelos de la marca Corvette
añadidos 7 modelos de la marca Cupra
añadidos 23 modelos de la marca Dacia
añadidos 13 modelos de la marca Daewoo
añadidos 6 modelos de la marca Daihatsu
añadidos 1 modelos de la marca DFSK
añadidos 5 modelos de la marca Dodge
añadidos 13 modelos de la marca DR
añadidos 11 modelos de la marca DS Automobiles
añadidos 2 modelos d

In [28]:
marcas_ordenadas = sorted([m['nombre'] for m in catalogo], key=len, reverse=True)

mapa_modelos = {m['nombre']: sorted(m['modelos'], key=len, reverse=True) for m in catalogo}

def normalizar(texto):
    if not texto: return ""
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto)
                  if unicodedata.category(c) != 'Mn')
    return texto.lower().strip()

marcas_normalizadas = {normalizar(m): m for m in marcas_ordenadas}

mapa_modelos_norm = {}
for marca_real, modelos in mapa_modelos.items():
    mapa_modelos_norm[normalizar(marca_real)] = {normalizar(mod): mod for mod in modelos}

In [21]:
driver = webdriver.Chrome()

In [32]:
lista_coches = []

In [33]:
for page in range(1, 341):
    url = f'{WEB_BASE}comprar-coche/?page={page}'
    
    try:
        driver.get(url)
        sleep(random.uniform(2, 4)) 

        cars = driver.find_elements(By.CLASS_NAME, 'root___Dz4kU')

        for car in cars:
            try:
                nombre_completo = car.find_element(By.CLASS_NAME, 'title___uRijL').text
                nombre_norm = normalizar(nombre_completo)

                marca_encontrada = "Desconocida"
                modelo_encontrado = "Desconocido"
                version = nombre_completo

                for m_norm, m_real in marcas_normalizadas.items():
                    if nombre_norm.startswith(m_norm):
                        marca_final = m_real
                        quitar_longitud = len(m_norm)
                        nombre_norm_sin_marca = nombre_norm[quitar_longitud:].strip()
                        version_final = nombre_completo[len(m_real):].strip()
                        
                        modelos_de_esta_marca = mapa_modelos_norm.get(m_norm, {})
                        modelos_ordenados = sorted(modelos_de_esta_marca.items(), key=lambda x: len(x[0]), reverse=True)
                        
                        for mod_norm, mod_real in modelos_ordenados:
                            if nombre_norm_sin_marca.startswith(mod_norm):
                                modelo_final = mod_real
                                version_final = version_final[len(mod_real):].strip()
                                break
                        break

                km = car.find_element(By.CSS_SELECTOR, '[data-qa-selector="mileage"]').text
                power = car.find_element(By.CSS_SELECTOR, '[data-qa-selector="horsePower"]').text
                price = car.find_element(By.CSS_SELECTOR, '[data-qa-selector="price"]').text

                
                km_limpio = km.split(' ')[0].replace('.', '')
                price_limpio = price.split(' ')[0].replace('.', '')

                datos_coche = {
                    "marca": marca_final,
                    "modelo": modelo_final,
                    "version": version_final,
                    "registration": car.find_element(By.CSS_SELECTOR, '[data-qa-selector="registration"]').text,
                    "km": int(km_limpio),
                    "gear_type": car.find_element(By.CSS_SELECTOR, '[data-qa-selector="transmission"]').text,
                    "fuel_type": car.find_element(By.CSS_SELECTOR, '[data-qa-selector="fuelType"]').text,
                    "power": power.split('(')[1].split(' ')[0],
                    "price": int(price_limpio)
                }
                
                lista_coches.append(datos_coche)

            except Exception as e:
                # Por si hay publicidad entre coches
                continue
                
    except Exception as e:
        print(f"Error en la página {page}: {e}")
        break

In [34]:
df = pd.DataFrame(lista_coches)
len(df)

3400

In [31]:
df

,marca,modelo,version,registration,km,gear_type,fuel_type,power,price
0,Alfa Romeo,Stelvio,2.2 JTDM,12/2021,118875,Automático,Diésel,190,24899
1,Hyundai,Tucson,1.6 T-GDI,12/2022,66838,Manual,Gasolina,150,23799
2,Mazda,CX-30,2.0 e-Skyactiv-X Mild-Hybrid,03/2022,69720,Manual,Gasolina,186,21499
3,Volkswagen,Tiguan,1.5 TSI ACT,08/2020,49789,Automático,Gasolina,150,26899
4,Citroen,C4,1.2 PureTech Mild-Hybrid,03/2025,14335,Automático,Gasolina,136,22299
5,Peugeot,208,1.2 PureTech,04/2019,67374,Manual,Gasolina,82,8499
6,Dacia,Sandero,1.0 TCe,10/2024,8709,Automático,Gasolina,91,18699
7,Citroen,C4 Cactus,1.2 e-THP,06/2019,11382,Manual,Gasolina,110,12399
8,Mercedes-Benz,C4 Cactus,Clase CLA CLA 200,07/2017,128485,Automático,Gasolina,155,20399
9,Peugeot,3008,1.2 Mild-Hybrid,03/2025,11383,Automático,Gasolina,136,29699


In [36]:
df.to_csv('../datasets/data_compramostucoche.csv', index=False)